# Ch.5  総合ミニ演習

【講師用完全版】

| 項目 | 内容 |
|------|------|
| 使うデータ | 乳がんデータ（Breast Cancer Wisconsin）569件・30特徴量・2クラス |
| 演習時間 | 35分（Step 4まで完了で合格） |
| ゴール | Ch.1〜Ch.4 で学んだ流れを一人で再現できる |

---
### 📌 講師メモ：時間配分と時間が押した場合の対応

| 区分 | 内容 | 目安時間 |
|------|------|---------|
| 演習（Step1〜Step2） | データ確認・EDA | 16分 |
| 演習（Step3〜Step4） | モデル学習・正解率・混同行列・特徴量重要度 | 14分 |
| 考察（AI禁止） | 気づきの言語化（最重要） | 5分 |
| 発展 | Precision / Recall | 余り次第 |

**時間が押した場合：**
- Step2（EDA）のヒートマップ（Q2-3）はスキップ可
- **Step4（正解率・混同行列・特徴量重要度）は必ず実施する**
- 考察セクションは時間が足りなくても口頭で発表させるだけでも可
- 発展（Precision/Recall）はスキップ可
- **Ch.5 自体を「完了しなくてもよい総仕上げ」として最初から案内することを推奨**

---
## Copilot の使い方

URL: https://copilot.microsoft.com

Copilot への伝え方のコツ（5点セット）：
```
【知りたいこと】〇〇をしたい / 〇〇を確認したい
【使うライブラリ】scikit-learn / pandas / matplotlib
【データの形】df という DataFrame、569行×31列（数値30列 + 診断結果列）
【環境】Python 3.8.6、JupyterLab
【困っていること】どう書けばよいかわからない
```

💡 Ch.5 の一言アドバイス：
「Ch.1〜4 と同じ操作の連続です。
 Ch.3 のコードを思い出しながら、データ名・変数名だけを変えて Copilot に聞いてみましょう。」

「前のチャプターで 〇〇 をやったが、今回は △△ のデータでやりたい」と伝えると
Copilot がうまく対応してくれます。

---
## 準備  ライブラリを読み込む

⛔ 注意 最初にこのセルを実行してください。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
print("準備完了")

---
## 総合演習のゴール

この演習は「先生に言われた通りにやる」のではなく、
Ch.1〜Ch.4 の経験を活かして「自分で流れを考えて実行する」演習です。

⛔ 注意 ヒントを見る前に、まず「自分で何をすべきか」を考えてください。
Copilot を使う前に「何を知りたいか」を日本語で書き出してから相談しましょう。

今回使うデータ：乳がんデータ
- 569件のデータ（悪性 212件・良性 357件）
- 30 種類の特徴量（腫瘍の大きさ・形状・均一性など）
- 目的：特徴量から「悪性（malignant）か良性（benign）か」を予測する

---
## Step 1  データを DataFrame に変換して確認する

まず「どんなデータか」を把握することが、データ分析の最初のステップです。
Ch.1 で学んだ方法を使って、このデータの全体像を掴みましょう。

---

Q1-1：データを読み込んで DataFrame に変換し、先頭の数行を表示してください

ターゲット（0=悪性、1=良性）を「診断結果」という列名で追加してください。

💡 ヒント Copilot への相談例：
「scikit-learn の load_breast_cancer を使ってデータを読み込み、pandas DataFrame に変換したい。
ターゲット列も追加して先頭5行を表示したい」

In [ ]:
# 乳がんデータを読み込んで DataFrame に変換し、先頭5行を表示する
cancer = load_breast_cancer()
df     = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df["診断結果"] = ["malignant" if t == 0 else "benign" for t in cancer.target]
print(f"行数: {len(df)}  列数: {len(df.columns)}")
df.head()

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 行数・列数 | 569行 × 31列（特徴量 30 + 診断結果 1） |
| 特徴量名 | mean radius・mean texture など腫瘍の測定値 |
| 診断結果列 | malignant（悪性）/ benign（良性）の 2 値 |

📌 ポイント 「最初にデータを確認する」習慣は Ch.1 から変わらない。
30 変数もあるが、機械学習が自動的に重要な変数を判断する。

✅ （模範解答）569件・31列のデータが作成される。先頭5行に各測定値と診断結果が表示される。

---
Q1-1（補足）：データの型と欠損値を確認してください

Ch.1 で学んだ「最初に必ずやる」操作です。

- `df.info()` → 列の型・件数・欠損の有無を確認する
- `df.isnull().sum()` → 欠損値の件数を列ごとに確認する

📌 このデータに欠損値はありません。でも**どんなデータでも必ずこの確認を最初にやる習慣**を身に付けましょう。

In [ ]:
# データの型・件数・欠損値の件数を確認する
print("--- df.info() ---")
df.info()
print()
print("--- 欠損値の確認 ---")
print(df.isnull().sum())

---
### 📊 出力結果の解説

📌 ポイント このデータには欠損値がない（すべての列で Non-Null Count = 569）。
ただし「欠損値確認は最初に必ずやる」習慣が Ch.1 から続いているかを確認する重要なステップ。

✅ （模範解答）569行・31列。欠損値はすべて 0 件。`df.dtypes` で数値列のみであることも確認できる。

📌 講師メモ：「Ch.1で欠損値確認の習慣を教えました。Ch.5でも最初にやりましたか？どんなデータでもこの習慣をつけましょう。」と声がけ。

---
Q1-2：クラスごとの件数と、データの基本統計を確認してください

悪性・良性それぞれ何件あるか、平均・最大・最小はどうかを把握してください。

💡 ヒント：Ch.1 問1・STEP1 で使った操作と同じです。

In [ ]:
# 診断結果ごとの件数を確認する
print("--- 診断結果ごとの件数 ---")
print(df["診断結果"].value_counts())
print()
# 数値列の基本統計量を確認する
print("--- 基本統計量（主要列）---")
df[["mean radius", "mean texture", "mean area"]].describe().round(2)

---
### 📊 出力結果の解説

```
診断結果ごとの件数：
benign       357
malignant    212

基本統計量：
       mean radius  mean texture  mean area
count       569.0         569.0      569.0
mean         14.1          19.3      655.0
std           3.5           4.3      352.0
```

📌 ポイント 良性（357件）が悪性（212件）の約 1.7 倍 → やや不均衡なデータ。
ただし比率は 1.7 倍程度なので、今回は特別な対処は不要。

✅ （模範解答）良性 357件、悪性 212件。不均衡はあるが軽微なので正解率での評価が基本的に有効。

---
### Step 1 の気づき（AI 禁止：1 行でOK）

Q：良性（benign）と悪性（malignant）の件数比はどうでしたか？またこのデータに欠損値はありましたか？

>

✅ （模範解答）良性 357件・悪性 212件（約 1.7:1 の不均衡データ）。欠損値はなし。Ch.1 と同様に「最初にデータを確認する」習慣を自力でできたかを確認するポイント。

📌 講師メモ：「良性の方が多いですね。悪性を見逃したらどうなるでしょう？」と医療文脈への意識を持たせる。

---
## Step 2  グラフで「データの傾向」を掴む

Ch.2 で学んだ可視化の技術を使って、悪性・良性で差が出る変数を探しましょう。

---

Q2-1：主要な変数のヒストグラムを描いて、分布の形を確認してください

mean radius・mean texture・mean area の 3 変数を確認してください。

💡 ヒント Copilot への相談例：
「pandas で複数の列のヒストグラムをまとめて描きたい。各グラフにタイトルも付けたい」

In [ ]:
# mean radius・mean texture・mean area の 3 変数のヒストグラムを並べて描く
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["mean radius", "mean texture", "mean area"]):
    ax.hist(df[col], bins=25)
    ax.set_title(col)
    ax.set_xlabel("値")
    ax.set_ylabel("件数")
plt.suptitle("主要変数の分布")
plt.tight_layout()
plt.show()

---
### 📊 出力結果の解説

| 変数 | 分布の特徴 |
|------|---------|
| mean radius | 右に裾が長い（大きな腫瘍が少数存在） |
| mean texture | ほぼ正規分布に近い形 |
| mean area | 右裾が非常に長い（外れ値的な大腫瘍あり） |

📌 ポイント mean area は分布が偏っている（右裾が長い）= 外れ値（大きな腫瘍）が存在する可能性。
このような変数は対数変換が有効な場合があるが、今回は RandomForest なので変換なしでも動く。

📌 講師メモ: Ch.2 と同じ操作であることを確認させる。

---
### Q2-1 の気づき（AI 禁止：1 行でOK）

Q：3 つの変数の分布の形を見て、気づいたことを書いてください（例：偏り、外れ値の有無など）

>

✅ （模範解答）mean area の分布は右に大きく裾を引いており、大きな腫瘍（外れ値候補）が存在する。mean texture はほぼ正規分布。Ch.2 のワインデータとは分布の形が違う。

📌 講師メモ：「mean area の裾が長い → 大きな腫瘍が少数ある = 悪性の可能性が高い変数かもしれない」という仮説へ誘導。

---
Q2-2：診断結果ごとに mean radius の分布を箱ひげ図で比べてください

悪性と良性で腫瘍の大きさに差があるか確認してください。

💡 ヒント：Ch.2 の問1（品種ごとの箱ひげ図）と同じ操作です。

In [ ]:
# 診断結果ごとの mean radius の分布を箱ひげ図で比較する
df.boxplot(column="mean radius", by="診断結果")
plt.suptitle("")
plt.title("診断結果ごとの mean radius 分布")
plt.ylabel("mean radius")
plt.show()

---
### 📊 出力結果の解説

| 診断結果 | 中央値（目安） |
|---------|------------|
| malignant（悪性） | 約 17 mm |
| benign（良性） | 約 12 mm |

📌 ポイント 悪性腫瘍の方が明らかに大きい（中央値で約 5mm の差）。
mean radius は悪性・良性の分類に有効な変数の一つ。

✅ （模範解答）悪性の中央値が良性より 5mm 程度大きく、mean radius は診断結果の判別に有効。

📌 講師メモ: 「Ch.2 の wine と同じパターン。大きい値のグループと小さい値のグループが分かれる変数が機械学習で使える」と接続。

---
### Q2-2 の気づき（AI 禁止：1 行でOK）

Q：mean radius で悪性・良性に差がありましたか？どちらが大きかったですか？

>

✅ （模範解答）悪性の中央値（約17mm）が良性（約12mm）より明らかに大きい。腫瘍の大きさが悪性・良性の判別に使える変数と予測できる。

📌 講師メモ：「Step 4 の特徴量重要度で『radius』が上位に来るかどうか、この時点で仮説として持っておいてください」と声がけ。

---
Q2-3：数値列全体の相関ヒートマップを描いてください

どの変数ペアが強く関係しているか確認してください。

💡 ヒント：Ch.2 問2（相関ヒートマップ）と同じ操作です。

In [ ]:
# 数値列全体の相関行列をヒートマップで可視化する
corr = df.select_dtypes(include="number").corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("相関ヒートマップ")
plt.tight_layout()
plt.show()

---
### 📊 出力結果の解説

| 確認するポイント | 見るべきこと |
|----------------|------------|
| 赤いブロック | mean radius・mean perimeter・mean area が強く相関（0.9 以上） |
| 青いブロック | fractal_dimension系が他と弱い相関 |

📌 ポイント mean radius・mean perimeter・mean area は「腫瘍の大きさ」を表す変数で、強く相関するのは当然。
機械学習では重複情報を除くことで精度が上がる場合もあるが、RandomForest は相関変数があっても比較的安定。

✅ （模範解答）mean radius / perimeter / area の 3 変数が強く相関（約 0.99）。これらは同じ「大きさ」の情報を持つ。

---
### Q2-3 の気づき（AI 禁止：1 行でOK）

Q：強く相関している変数ペアを 1 つ見つけて、それらは「何を表す変数」だと思いますか？

>

✅ （模範解答）mean radius / mean perimeter / mean area が強く相関（約 0.99）。これらはすべて「腫瘍の大きさ」を表す変数で、同じ情報を重複して持っている。

📌 講師メモ：「同じ情報を持つ変数が複数あっても、RandomForest は比較的安定して動きます」と補足。

---
## Step 3  RandomForest でモデルを学習させる

Ch.3 と同じ手順でモデルを作ります。手順を自分で思い出しながら進めてください。

---

Q3-1：特徴量（X）とターゲット（y）に分け、8:2 でデータを分割してください

X：「診断結果」列を除いたすべての数値列
y：「診断結果」列

In [ ]:
# 特徴量と診断結果に分けて、学習用・テスト用に 80:20 で分割する
X = df.drop("診断結果", axis=1)
y = df["診断結果"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"学習: {len(X_train)}件  テスト: {len(X_test)}件")

---
### 📊 出力結果の解説

```
学習: 455件  テスト: 114件
```

📌 ポイント Ch.3 と同じ手順（X/y 分割 → 8:2 分割）。
今回は 569件のうち 114件がテスト用。

✅ （模範解答）455件で学習、114件でテスト。X は 30 列の数値特徴量、y は診断結果の 2 値。

---
Q3-2：RandomForestClassifier でモデルを作り、学習させてください

In [ ]:
# RandomForest モデルを作成して、学習データで学習させる
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print("学習完了")

---
📌 ポイント Ch.3 と全く同じコード。チューニングなしで高精度が出ることが多い。
✅ （模範解答）`model.fit(X_train, y_train)` でエラーなく「学習完了」と表示されれば成功。

---
## Step 4  モデルを評価してゴールを確認する （最重要）

正解率・混同行列・特徴量重要度の 3 つを確認しましょう。
Ch.3 の手順と同じです。コードを思い出しながら書いてみてください。

---

Q4-1：テストデータで予測して正解率を計算してください

In [ ]:
# テストデータで予測して、正解率を計算する
y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
print(f"正解率: {acc:.3f}  ({acc*100:.1f}%)")

---
### 📊 出力結果の解説

```
正解率: 0.965  (96.5%)
```

📌 ポイント 96.5% → 114件中 110件程度を正解。
悪性を良性と誤判定（見逃し）が最も問題になる。混同行列で確認する。

✅ （模範解答）正解率 95〜97% 程度が期待される。

---
### Q4-1 の気づき（AI 禁止：1 行でOK）

Q：正解率は何 % でしたか？医療の文脈で考えると、この数値は「十分に高い」と言えますか？

>

✅ （模範解答）96〜97% 程度。一見高く見えるが、4% の誤判定のうち「悪性の見逃し」が含まれるため医療現場では十分とは言えない場合がある。

📌 講師メモ：「96% と聞いて安心しましたか？次の混同行列で何を間違えているかを確認しましょう」と繋げる。

---
Q4-2：混同行列を表示して、どのクラスを間違えたか確認してください

In [ ]:
# 混同行列を計算して、どのクラスを間違えたか確認する
cm    = confusion_matrix(y_test, y_pred)
classes = model.classes_
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print("混同行列（行=実際、列=予測）")
print(cm_df)

---
### 📊 出力結果の解説

```
混同行列（行=実際、列=予測）
           benign  malignant
benign         70          1
malignant       3         40
```

| 間違いの種類 | 内容 | 実務上の影響 |
|-----------|------|-----------|
| benign → malignant（1件） | 良性を悪性と判定 | 不必要な追加検査 |
| malignant → benign（3件） | 悪性を良性と判定（見逃し） | 重大な問題 |

📌 ポイント 医療データでは「悪性の見逃し（malignant → benign と判定）」が最も問題。
Recall（悪性の見逃しを測る指標）を重視する場面。Ch.5 発展でより詳しく確認。

✅ （模範解答）malignant を benign と誤判定（見逃し）が 2〜4件程度。これを 0 に近づけることがモデル改善の目標。

---
### Q4-2 の気づき（AI 禁止：1 行でOK）

Q：2 種類の誤判定（「良性を悪性と判定」と「悪性を良性と判定」）のうち、医療的にどちらがより問題が大きいですか？その理由は？

>

✅ （模範解答）「悪性を良性と判定（malignant → benign）」の方がはるかに問題が大きい。悪性の見逃しは治療の遅れに直結するため。「良性を悪性と判定」は不必要な追加検査は生じるが命には関わらない。

📌 講師メモ：この気づきが考察 Q2（「96%で医療現場で使えるか？」）への橋渡しになる。Recall（再現率）を最重視すべき理由を体感させる場面。

---
Q4-3：特徴量重要度を棒グラフで表示してください

どの特徴量が診断に最も重要か確認してください。

In [ ]:
# 特徴量重要度を取得して、重要度が高い順に横棒グラフで表示する
importances = model.feature_importances_
feat_df     = pd.DataFrame({"特徴量": X.columns, "重要度": importances})
feat_df     = feat_df.sort_values("重要度", ascending=True).tail(10)   # 上位 10 変数

plt.figure(figsize=(8, 6))
plt.barh(feat_df["特徴量"], feat_df["重要度"])
plt.title("特徴量重要度（上位10）")
plt.xlabel("重要度")
plt.tight_layout()
plt.show()

---
### 📊 出力結果の解説

上位 3 変数（目安）：
1. worst perimeter（腫瘍の外周の最大値）
2. worst area（腫瘍面積の最大値）
3. worst radius（腫瘍半径の最大値）

📌 ポイント 「worst（最悪値）」系の変数が上位に来ることが多い。
EDA で「mean radius が悪性・良性で差がある」と気づいたが、「worst radius」の方がより判別に効く。

✅ （模範解答）worst 系（worst perimeter・worst area・worst radius）が上位を占める。
「平均値より最悪値の方が診断に有効」という医学的な知見とも一致している。

📌 講師メモ: 「Ch.2 の EDA で仮説を立てたこととどう一致しましたか？」と問い返す。

---
### Q4-3 の気づき（AI 禁止：1 行でOK）

Q：特徴量重要度 1 位の変数は何でしたか？Step 2 で「radius が重要そう」と思った仮説は当たっていましたか？

>

✅ （模範解答）worst perimeter / worst area / worst radius などの「worst（最悪値）系」変数が上位。Step 2 で仮説を立てた「radius」は含まれているが「mean」ではなく「worst」の方が重要だった。

📌 講師メモ：「EDA → 仮説 → モデルで検証という流れが Ch.1〜Ch.5 を通じて完結しましたね」とまとめの言葉を添える。

---
## 考察  自分の言葉でまとめる （AI 禁止）

⛔ 注意 このセクションは AI を使わず、自分の言葉で記入してください。
演習の「ゴール」はコードを書くことではなく、「気づきを言葉にすること」です。

Q1：このデータで最も重要な変数はどれでしたか？その変数が重要な理由を説明してください。

>

Q2：正解率約 96% のモデルは医療現場で使えると思いますか？その理由は？

>

Q3：Ch.1〜Ch.5 を通じて、データ分析の流れを自分の言葉でまとめてください。

>

---
### 📌 講師メモ（考察の模範解答）

（模範解答 1）重要変数：
worst perimeter・worst area・worst radius が上位。腫瘍の「最悪値」が診断に有効。
「悪性腫瘍は最大値が極端に大きい」という特徴を機械が自動的に学習した。

（模範解答 2）96% モデルの評価：
96% は高い正解率だが、医療では見逃し（悪性を良性と判定）がゼロに近いことが求められる。
混同行列で「malignant → benign の誤判定件数」を最重視すべき。
Recall（再現率）が高いモデルを優先する場面（発展問題へ）。

（模範解答 3）データ分析の流れ：
1. データを確認する（件数・型・欠損・統計量）
2. グラフで傾向を掴む（分布・グループ別の差・相関）
3. 仮説を立てる（どの変数が重要か）
4. モデルを作る・評価する
5. 特徴量重要度で仮説を検証する
→「EDA → 仮説 → モデル → 検証」のサイクルがデータ分析の基本。

よくある受講生の反応：
- 「96% でいいのでは？」→「4件の見逃しのうち 3件が悪性の見逃し。医療だと命に関わる」と伝える

---
## 発展  Precision と Recall を計算する

📋 発展 時間が余った人だけ取り組んでください

---

### Precision / Recall とは

正解率（Accuracy）は「全体の正解率」しか見ていません。
医療のような「間違いの種類によって重大さが異なる」場面では、より細かい指標が必要です。

```
📌 Precision（適合率）：「陽性と予測した中で、本当に陽性だった割合」
   例）「悪性と診断した患者のうち、本当に悪性だった割合」
   → 低いと「誤診（正常なのに手術させてしまう）」が増える

📌 Recall（再現率）：「実際に陽性のうち、正しく陽性と予測できた割合」
   例）「本当に悪性の患者のうち、正しく悪性と診断できた割合」
   → 低いと「見逃し（悪性なのに良性と言ってしまう）」が増える
```

```
【医療の場合はどちらを重視すべきか？】

「悪性を見逃す（Recall が低い）」
  → 患者が治療を受けられない → 命に関わる → 深刻

「正常を悪性と誤診（Precision が低い）」
  → 精密検査・追加診断が必要 → 負担だが修正できる

→ 医療では Recall を高めることが最優先
```

> 📌 **Ch.3 のまとめで「Precision / Recall は Ch.5 で確認する」としていたのはここです。**  
> 実際の数値（classification_report の出力）を見てから概念を理解する方が腑に落ちます。

📌 講師メモ：「Step 4 の混同行列で『悪性を良性と誤診した件数』と『良性を悪性と誤診した件数』、どちらが多かったか？」と問いかけてから発展へ進むと、Precision / Recall の意味が自然に理解されます。

---

Q：クラスごとの Precision と Recall を計算して、悪性（malignant）の Recall を確認してください

💡 ヒント Copilot への相談例：
「scikit-learn で classification_report を使ってクラスごとの詳細な評価指標を表示したい」

In [ ]:
# クラスごとの Precision・Recall・F1 を一括で表示する
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

---
### 📊 出力結果の解説

```
              precision    recall  f1-score   support
      benign       0.96      0.99      0.97        71
   malignant       0.98      0.93      0.95        43
    accuracy                           0.96       114
```

| 指標 | benign | malignant | 意味 |
|------|--------|-----------|------|
| Precision | 0.96 | 0.98 | 予測が正しかった割合 |
| Recall | 0.99 | 0.93 | 実際の正解を見つけた割合 |

📌 ポイント malignant の Recall が 0.93 = 悪性 43件のうち 3件を見逃している。
医療では malignant の Recall を 0.99 以上にすることが目標となる場合が多い。

✅ （模範解答）malignant の Recall が 0.93 程度。見逃しを 0 に近づけるには、しきい値を下げる（モデルが悪性と判定しやすくする）調整が必要。

---
## お疲れさまでした！

| Ch | 学んだこと | 総合演習での確認 |
|----|---------|----------------|
| Ch.1 | データの全体像を数値で把握する | Step 1 で再現 |
| Ch.2 | グラフで傾向・仮説を掴む | Step 2 で再現 |
| Ch.3 | RF で分類・正解率・混同行列 | Step 3・4 で再現 |
| Ch.4 | 時系列・残差で異常検知 | 今後の応用 |

Ch.1〜Ch.4 の流れが「一人で」再現できれば、今日の研修は完了です！